# Colab Part D: fundamentals backfill (CSE report archive + SEC EDGAR)

**J26-SE-325, Component 1.** Run this **first**: Parts A (forecaster head) and C (Laya) both
train on its output.

The local link (~24 KB/s) cannot download ~282 companies x up to ~70 report PDFs. Colab can.
This notebook runs `experiments/fundamentals_backfill.py`, which:

1. lists every CSE voting / non-voting security, fetches its report index, downloads each
   interim (and optionally annual) PDF, extracts the statements with the same extractor the
   service uses (`fundamentals/extract.py`), runs the consistency checks, and computes ratios,
   F-score and Z''-score **as of the report's upload date**;
2. pulls SEC EDGAR XBRL facts for the US equities in `configs/universe.yaml` (full filing
   history with filing dates; yfinance only has ~5 quarters);
3. exports one small parquet to bring home.

**No GPU needed** (CPU runtime is fine). **Resumable:** the checkpoint lives on Google Drive,
so after a disconnect just run all cells again; finished reports are skipped.

In [ ]:
# === BOOTSTRAP (same contract as colab_finetune.ipynb) ================================
# Clone or update the branch, chdir into the component, install PIP_PKGS. Safe to re-run.
REPO_URL = "https://github.com/SE-Y4S1/J26-SE-325.git"
BRANCH = "Nivakaran"  # the component is NOT on main
SUBDIR = "backend/Portfolio-Optimization"
PIP_PKGS = "pypdf yfinance pandas pyyaml requests pydantic pandas-ta-openbb"

import importlib, os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
OWN_PACKAGES = {"agent", "configs", "data", "evaluation", "experiments", "features",
                "forecasting", "fundamentals", "news", "optimization", "decision", "service"}


def find_root(start):
    for base in (start, *start.parents):
        if (base / "configs/universe.yaml").exists() and (base / "fundamentals").exists():
            return base
    return None


# In Colab always fetch: a re-run starts INSIDE the checkout, so find_root
# succeeded and pushed fixes never reached a live runtime.
root = None if IN_COLAB else find_root(Path.cwd())
if root is None:
    if not IN_COLAB:
        raise SystemExit("cd into backend/Portfolio-Optimization first.")
    checkout = Path("/content/J26-SE-325")
    if (checkout / SUBDIR).exists():
        if subprocess.run(["git", "-C", str(checkout), "fetch", "--depth", "1", "origin", BRANCH]).returncode == 0:
            subprocess.run(["git", "-C", str(checkout), "reset", "--hard", "FETCH_HEAD"])
    else:
        if subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(checkout)]).returncode:
            from getpass import getpass
            token = getpass("Clone failed. GitHub token (repo scope), blank to abort: ").strip()
            if not token:
                raise SystemExit("Cannot reach " + REPO_URL)
            subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                            REPO_URL.replace("https://", "https://" + token + "@"), str(checkout)], check=True)
    root = checkout / SUBDIR

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
for _name in [n for n in sys.modules if n.split(".")[0] in OWN_PACKAGES]:
    del sys.modules[_name]
importlib.invalidate_caches()
if IN_COLAB and PIP_PKGS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + PIP_PKGS.split(), check=True)
print("OK root=" + str(root))

## Checkpoint on Drive

Colab's local disk is wiped on disconnect. Keeping the checkpoint on Drive is what makes
the run resumable across sessions.

In [ ]:
from pathlib import Path

OUT = Path("artifacts/backfill")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/j26_se_325_backfill")
OUT.mkdir(parents=True, exist_ok=True)
print("checkpoint dir:", OUT)

## SEC contact

SEC EDGAR rejects requests without a descriptive User-Agent that includes a contact address.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("SEC_USER_AGENT"):
    os.environ["SEC_USER_AGENT"] = "J26-SE-325 research " + input("Contact email for SEC User-Agent: ").strip()
print(os.environ["SEC_USER_AGENT"])

## Smoke run (3 CSE companies, no US)

Every failure mode (CSE endpoint shape, CDN path, PDF parsing) shows up here in a few
minutes rather than an hour into the full run.

In [ ]:
from experiments.fundamentals_backfill import main

main(["--out", str(OUT / "smoke"), "--cse-limit", "3", "--no-us"])

import pandas as pd
smoke = pd.read_parquet(OUT / "smoke" / "fundamentals_backfill.parquet")
print(len(smoke), "records")
smoke.head()

## Full run

Interims for every CSE company plus EDGAR for the US names. Several hours; re-run this
cell after a disconnect and it continues. Add `"--include-annual"` for annual reports too
(much larger PDFs; the Q4 interim already carries the year-end figures).

In [ ]:
main(["--out", str(OUT)])

## Extraction quality summary

Share of reports usable (all identity checks passed or confidence >= 0.5), by template.
Low rates for a template mean its synonym list needs extending in `fundamentals/extract.py`.

In [ ]:
import json
import pandas as pd

table = pd.read_parquet(OUT / "fundamentals_backfill.parquet")
records = table["record"].map(json.loads)
summary = pd.DataFrame({
    "template": records.map(lambda r: r.get("template")),
    "usable": records.map(lambda r: bool(r.get("usable"))),
    "confidence": records.map(lambda r: r.get("confidence")),
    "source": records.map(lambda r: r.get("source")),
})
print(summary.groupby(["source", "template"]).agg(reports=("usable", "size"), usable_rate=("usable", "mean"),
                                                   mean_confidence=("confidence", "mean")))

## Bring it home

A few MB. Then, on the local machine:

```bash
python -m fundamentals.pipeline --import fundamentals_backfill.parquet
```

Parts A and C read the same file directly on Colab (from Drive), so keep it there too.

In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download(str(OUT / "fundamentals_backfill.parquet"))